In [1]:
import os
import shutil
import pandas as pd
import subprocess

# Load the original Excel file
original_csv = "E:/Datasets/manifest-21CCx8fx964169794518489879/metadata.csv"
metadata = pd.read_csv(original_csv)

# Base directories
dicom_base_dir = "E:/Datasets/manifest-21CCx8fx964169794518489879/TCGA-GBM"
nifti_base_dir = "E:/Datasets/Selected Data"
os.makedirs(nifti_base_dir, exist_ok=True)

# Initialize a list to store selected data
selected_data = []

# Function to convert DICOM to NIFTI using dcm2niix
def convert_dicom_to_nifti(dicom_dir, nifti_dir, output_name):
    os.makedirs(nifti_dir, exist_ok=True)
    command = f"dcm2niix -o {nifti_dir} -f {output_name} {dicom_dir}"
    subprocess.run(command, shell=True, check=True)
    # Find the generated NIFTI file
    for file in os.listdir(nifti_dir):
        if file.startswith(output_name) and file.endswith(".nii.gz"):
            return os.path.join(nifti_dir, file)
    return None

# Iterate through each subject
for subject_id, subject_data in metadata.groupby("Subject ID"):
    print(f"Processing {subject_id}...")

    # Create subject directories
    subject_nifti_dir = os.path.join(nifti_base_dir, f"{subject_id}")
    os.makedirs(subject_nifti_dir, exist_ok=True)

    # Select T1 (or FSPGR)
    t1_data = subject_data[subject_data["Series Description"].str.contains("T1|FSPGR", case=False, na=False)]
    if not t1_data.empty:
        t1_series = t1_data.iloc[0]  # Select the first T1/FSPGR series
        t1_dicom_dir = t1_series["File Location"]
        t1_nifti_dir = os.path.join(subject_nifti_dir, "T1")
        t1_nifti_path = convert_dicom_to_nifti(t1_dicom_dir, t1_nifti_dir, "t1")
    else:
        print(f"No T1/FSPGR found for {subject_id}. Skipping...")
        continue

    # Select T2 (or FLAIR)
    t2_data = subject_data[subject_data["Series Description"].str.contains("T2|FLAIR", case=False, na=False)]
    if not t2_data.empty:
        t2_series = t2_data.iloc[0]  # Select the first T2/FLAIR series
        t2_dicom_dir = t2_series["File Location"]
        t2_nifti_dir = os.path.join(subject_nifti_dir, "T2")
        t2_nifti_path = convert_dicom_to_nifti(t2_dicom_dir, t2_nifti_dir, "t2")
    else:
        print(f"No T2/FLAIR found for {subject_id}. Skipping T2...")
        t2_nifti_path = None

    # Add selected data to the list
    selected_data.append({
        "Subject ID": subject_id,
        "T1 Series ID": t1_series["Series UID"] if not t1_data.empty else None,
        "T1 File Path": t1_nifti_path if not t1_data.empty else None,
        "T2 Series ID": t2_series["Series UID"] if not t2_data.empty else None,
        "T2 File Path": t2_nifti_path if not t2_data.empty else None,
    })

    # Save metadata for the subject
    subject_metadata = pd.DataFrame([{
        "Subject ID": subject_id,
        "T1 Series ID": t1_series["Series ID"] if not t1_data.empty else None,
        "T1 File Path": t1_nifti_path if not t1_data.empty else None,
        "T2 Series ID": t2_series["Series ID"] if not t2_data.empty else None,
        "T2 File Path": t2_nifti_path if not t2_data.empty else None,
    }])
    subject_metadata.to_csv(os.path.join(subject_nifti_dir, "metadata.csv"), index=False)

# Save the selected data to a new Excel file
selected_df = pd.DataFrame(selected_data)
selected_df.to_excel(os.path.join(nifti_base_dir, "selected_data.xlsx"), index=False)

print("Dataset organization and conversion complete!")

Processing TCGA-02-0003...


CalledProcessError: Command 'dcm2niix -o E:/Datasets/Selected Data\TCGA-02-0003\T1 -f t1 .\TCGA-GBM\TCGA-02-0003\06-08-1997-NA-MRI BRAIN WWO CONTRAMR-81239\13.000000-AX T1 POST-19694' returned non-zero exit status 5.